In [1]:
# ============================================================
# EXP-3) Constant-rate supervision risk defense (TEST-only perturbations)
#   - Dataset: MHEALTH (6 activities): act_id = [6,7,8,10,11,12]
#   - LOSO (train on 9 subjects, test on 1 subject) per activity
#   - Compare: OURS vs B0
#       * SAME windowing, SAME count-only supervision (window y_count), SAME pipeline
#       * Only differences for B0:
#           - loss = terminal count regression per window (no recon/smooth/phase/effK)
#           - backbone capacity increased: hidden_dim*2, encoder/decoder layers +1
#   - TEST perturbations (no retraining):
#       1) orig
#       2) pause_insert (insert zeros in the middle; duration increases)
#       3) gradual_tempo_drift (linear time-warp; gradually slower/faster across time)
#   - Output: print summaries only (no CSV saving, no visualization)
# ============================================================

import os
import glob
import random
import zlib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ---------------------------------------------------------------------
# 1) Strict Seeding
# ---------------------------------------------------------------------
def set_strict_seed(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def _stable_seed_from_text(text: str, base_seed: int = 0) -> int:
    h = zlib.adler32(text.encode("utf-8"))
    return int((base_seed + h) % (2**31 - 1))


# ---------------------------------------------------------------------
# 2) Data Loading
# ---------------------------------------------------------------------
def load_mhealth_dataset(data_dir, target_activities_map, column_names):
    full_dataset = {}
    file_list = sorted(glob.glob(os.path.join(data_dir, "mHealth_subject*.log")))
    if not file_list:
        print(f"[Warning] No mHealth logs found in {data_dir}")
        return {}

    for file_path in file_list:
        file_name = os.path.basename(file_path)
        subj_part = file_name.split('.')[0]
        try:
            subj_id_num = int(''.join(filter(str.isdigit, subj_part)))
            subj_key = f"subject{subj_id_num}"
        except:
            subj_key = subj_part

        try:
            df = pd.read_csv(file_path, sep="\t", header=None)
            df = df.iloc[:, :len(column_names)]
            df.columns = column_names

            subj_data = {}
            for label_code, activity_name in target_activities_map.items():
                activity_df = df[df['activity_id'] == label_code].copy()
                if not activity_df.empty:
                    subj_data[activity_name] = activity_df.drop(columns=['activity_id'])
            full_dataset[subj_key] = subj_data
        except Exception as e:
            print(f"Error loading {file_name}: {e}")
            pass

    return full_dataset


def prepare_trial_list(label_config, full_data, target_map, feature_map):
    """
    label_config: list of (subj, act_id, gt_count)
    returns list of dict:
      data: (T,C) normalized per trial
      count: float (trial total count)
      subj, act_id, act_name, meta
    """
    trial_list = []
    for subj, act_id, gt_count in label_config:
        act_id = int(act_id)
        act_name = target_map.get(act_id)
        feats = feature_map.get(act_id)

        if subj in full_data and act_name in full_data[subj]:
            raw_df = full_data[subj][act_name][feats]
            raw_np = raw_df.values.astype(np.float32)

            # trial-wise z-score (MODEL INPUT)
            mean = raw_np.mean(axis=0)
            std = raw_np.std(axis=0) + 1e-6
            norm_np = (raw_np - mean) / std

            trial_list.append({
                "data": norm_np,
                "count": float(gt_count),
                "subj": subj,
                "act_id": act_id,
                "act_name": act_name,
                "meta": f"{subj}_{act_name}",
            })
        else:
            print(f"[Skip] Missing data for {subj} - act_id={act_id} ({act_name})")

    return trial_list


# ---------------------------------------------------------------------
# 3) Windowing (TRAIN) — count-only supervision (window y_count)
# ---------------------------------------------------------------------
def trial_list_to_windows(trial_list, fs, win_sec=8.0, stride_sec=4.0, drop_last=True):
    """
    TRAIN: trial -> windows
    window label = trial avg rate * window duration
    """
    win_len = int(round(win_sec * fs))
    stride = int(round(stride_sec * fs))
    assert win_len > 0 and stride > 0

    windows = []
    for item in trial_list:
        x = item["data"]  # (T,C)
        T = x.shape[0]
        total_count = float(item["count"])
        meta = item["meta"]

        total_dur = max(T / float(fs), 1e-6)
        rate_trial = total_count / total_dur

        if T < win_len:
            win_dur = T / float(fs)
            windows.append({
                "data": x,
                "count": rate_trial * win_dur,
                "meta": f"{meta}__win[0:{T}]",
                "length": T,
            })
            continue

        last_start = T - win_len
        starts = list(range(0, last_start + 1, stride))

        for st in starts:
            ed = st + win_len
            win_dur = win_len / float(fs)
            windows.append({
                "data": x[st:ed],
                "count": rate_trial * win_dur,
                "meta": f"{meta}__win[{st}:{ed}]",
                "length": win_len,
            })

        if not drop_last:
            last_st = starts[-1] + stride
            if last_st < T:
                ed = T
                win_dur = (ed - last_st) / float(fs)
                windows.append({
                    "data": x[last_st:ed],
                    "count": rate_trial * win_dur,
                    "meta": f"{meta}__win[{last_st}:{ed}]",
                    "length": (ed - last_st),
                })

    return windows


# ---------------------------------------------------------------------
# 4) Windowing inference (TEST): trial -> windows -> mean rate -> total count
# ---------------------------------------------------------------------
def predict_count_by_windowing(model, x_np, fs, win_sec, stride_sec, device,
                              tau=1.0, batch_size=64):
    win_len = int(round(win_sec * fs))
    stride = int(round(stride_sec * fs))
    T = x_np.shape[0]
    total_dur = T / float(fs)

    if T <= win_len:
        x_tensor = torch.tensor(x_np, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(device)  # (1,C,T)
        with torch.no_grad():
            rate_hat = model.forward_rate(x_tensor, mask=None, tau=tau)
        return float(rate_hat.item() * total_dur)

    starts = list(range(0, T - win_len + 1, stride))
    windows = np.stack([x_np[st:st + win_len] for st in starts], axis=0)  # (N,W,C)
    xw = torch.tensor(windows, dtype=torch.float32).permute(0, 2, 1).to(device)  # (N,C,W)

    rates = []
    model.eval()
    with torch.no_grad():
        for i in range(0, xw.shape[0], batch_size):
            xb = xw[i:i + batch_size]
            r_hat = model.forward_rate(xb, mask=None, tau=tau)  # (B,)
            rates.append(r_hat.detach().cpu().numpy())

    rates = np.concatenate(rates, axis=0)
    rate_mean = float(rates.mean())
    return float(rate_mean * total_dur)


# ---------------------------------------------------------------------
# 5) TEST perturbations
# ---------------------------------------------------------------------
def perturb_pause_insert(x_np: np.ndarray, fs: int, pause_sec: float = 2.0,
                         center_frac: float = 0.5) -> np.ndarray:
    """
    Insert a zero-signal pause segment in the middle.
    - duration increases, GT count stays same (pause adds time with no reps)
    """
    x = np.asarray(x_np, dtype=np.float32)
    T, C = x.shape
    pause_len = int(round(float(pause_sec) * float(fs)))
    pause_len = max(pause_len, 1)

    mid = int(round(float(center_frac) * float(T)))
    mid = np.clip(mid, 0, T)

    z = np.zeros((pause_len, C), dtype=np.float32)
    out = np.concatenate([x[:mid], z, x[mid:]], axis=0)
    return out


def perturb_gradual_tempo_drift(x_np: np.ndarray, drift_strength: float = 0.35,
                                direction: str = "slow") -> np.ndarray:
    """
    Gradual time-warp (non-uniform resampling):
      - 'slow': progressively slows down toward the end (more samples allocated later)
      - 'fast': progressively speeds up toward the end (more samples allocated earlier)
    Implemented by warping normalized time grid with a power curve.
    """
    x = np.asarray(x_np, dtype=np.float32)
    T, C = x.shape
    if T < 4:
        return x.copy()

    t_old = np.linspace(0.0, 1.0, T, dtype=np.float32)

    s = float(np.clip(drift_strength, 0.0, 0.95))
    # power exponent in [~0.5, ~2.0]
    if direction == "slow":
        p = 1.0 + 2.0 * s     # >1 pushes more resolution to early? Actually u=t^p compresses later.
        # For "slow later", want expanded later => u = t^(1/p) (p>1)
        u = np.power(t_old, 1.0 / max(p, 1e-6)).astype(np.float32)
    else:
        p = 1.0 + 2.0 * s
        # "fast later" => compress early, expand early => u = t^p
        u = np.power(t_old, p).astype(np.float32)

    # Keep length same (T) but change sampling positions
    out = np.zeros_like(x)
    for c in range(C):
        out[:, c] = np.interp(t_old, u, x[:, c]).astype(np.float32)
    return out


# ---------------------------------------------------------------------
# 6) Dataset / Collate
# ---------------------------------------------------------------------
class WindowDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        item = self.windows[idx]
        data = torch.tensor(item['data'], dtype=torch.float32).transpose(0, 1)  # (C,T)
        count = torch.tensor(item['count'], dtype=torch.float32)
        return data, count, item['meta']


def collate_variable_length(batch):
    max_len = max([x[0].shape[1] for x in batch])
    C = batch[0][0].shape[0]

    padded_data, masks, counts, metas, lengths = [], [], [], [], []
    for data, count, meta in batch:
        T = data.shape[1]
        lengths.append(T)

        pad_size = max_len - T
        if pad_size > 0:
            pad = torch.zeros(C, pad_size)
            d_padded = torch.cat([data, pad], dim=1)
            mask = torch.cat([torch.ones(T), torch.zeros(pad_size)], dim=0)
        else:
            d_padded = data
            mask = torch.ones(T)

        padded_data.append(d_padded)
        masks.append(mask)
        counts.append(count)
        metas.append(meta)

    return {
        "data": torch.stack(padded_data),
        "mask": torch.stack(masks),
        "count": torch.stack(counts),
        "length": torch.tensor(lengths, dtype=torch.float32),
        "meta": metas
    }


# ---------------------------------------------------------------------
# 7) Models (OURS / B0)
# ---------------------------------------------------------------------
class ManifoldEncoder(nn.Module):
    def __init__(self, input_ch, hidden_dim=128, latent_dim=16, n_layers=2):
        super().__init__()
        layers = []
        ch_in = input_ch
        for li in range(n_layers):
            layers += [
                nn.Conv1d(ch_in, hidden_dim, 5, padding=2),
                nn.ReLU(),
            ]
            ch_in = hidden_dim
        layers += [nn.Conv1d(hidden_dim, latent_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        z = self.net(x)            # (B,D,T)
        z = z.transpose(1, 2)      # (B,T,D)
        return z


class ManifoldDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, out_ch, n_layers=2):
        super().__init__()
        layers = []
        ch_in = latent_dim
        for li in range(n_layers):
            layers += [
                nn.Conv1d(ch_in, hidden_dim, 5, padding=2),
                nn.ReLU(),
            ]
            ch_in = hidden_dim
        layers += [nn.Conv1d(hidden_dim, out_ch, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        zt = z.transpose(1, 2)     # (B,D,T)
        x_hat = self.net(zt)       # (B,C,T)
        return x_hat


class MultiRateHead(nn.Module):
    def __init__(self, latent_dim=16, hidden=64, K_max=6):
        super().__init__()
        self.K_max = K_max
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1 + K_max)  # [amp | phase_logits...]
        )

    def forward(self, z, tau=1.0):
        out = self.net(z)                     # (B,T,1+K)
        amp = F.softplus(out[..., 0])         # (B,T) >=0
        phase_logits = out[..., 1:]           # (B,T,K)
        phase = F.softmax(phase_logits / tau, dim=-1)
        return amp, phase, phase_logits


class OursKAutoCount(nn.Module):
    """
    OURS: window-stabilized rate learning with aux losses
    Output: avg_rep_rate (reps/sec)
    """
    def __init__(self, input_ch, hidden_dim=128, latent_dim=16, K_max=6,
                 enc_layers=2, dec_layers=2):
        super().__init__()
        self.encoder = ManifoldEncoder(input_ch, hidden_dim, latent_dim, n_layers=enc_layers)
        self.decoder = ManifoldDecoder(latent_dim, hidden_dim, input_ch, n_layers=dec_layers)
        self.rate_head = MultiRateHead(latent_dim, hidden=hidden_dim, K_max=K_max)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
        with torch.no_grad():
            b = self.rate_head.net[-1].bias
            b.zero_()
            b[0].fill_(-2.0)  # amp bias

    @staticmethod
    def _masked_mean_time(x, mask=None, eps=1e-6):
        if mask is None:
            return x.mean(dim=1)
        if x.dim() == 2:
            m = mask.to(dtype=x.dtype, device=x.device)
            return (x * m).sum(dim=1) / (m.sum(dim=1) + eps)
        elif x.dim() == 3:
            m = mask.to(dtype=x.dtype, device=x.device).unsqueeze(-1)
            return (x * m).sum(dim=1) / (m.sum(dim=1) + eps)
        else:
            raise ValueError(f"Unsupported dim for masked mean: {x.dim()}")

    def forward(self, x, mask=None, tau=1.0):
        z = self.encoder(x)              # (B,T,D)
        x_hat = self.decoder(z)          # (B,C,T)

        amp_t, phase_p, _ = self.rate_head(z, tau=tau)
        micro_rate_t = amp_t             # (B,T)

        p_bar = self._masked_mean_time(phase_p, mask)           # (B,K)
        k_hat = 1.0 / (p_bar.pow(2).sum(dim=1) + 1e-6)          # (B,)

        rep_rate_t = micro_rate_t / (k_hat.unsqueeze(1) + 1e-6) # (B,T)
        if mask is not None:
            rep_rate_t = rep_rate_t * mask

        if mask is None:
            avg_rep_rate = rep_rate_t.mean(dim=1)
        else:
            avg_rep_rate = (rep_rate_t * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-6)

        aux = {"phase_p": phase_p, "rep_rate_t": rep_rate_t, "k_hat": k_hat, "x_hat": x_hat}
        return avg_rep_rate, aux

    def forward_rate(self, x, mask=None, tau=1.0):
        r, _ = self.forward(x, mask=mask, tau=tau)
        return r


class B0TerminalRegressor(nn.Module):
    """
    B0: same windowing/count-only supervision, but terminal count regression per window.
    - Bigger backbone: hidden_dim*2, encoder/decoder layers +1
    - Predict window_count directly (>=0 via softplus)
    """
    def __init__(self, input_ch, hidden_dim=256, latent_dim=16, enc_layers=3, dec_layers=3):
        super().__init__()
        self.encoder = ManifoldEncoder(input_ch, hidden_dim, latent_dim, n_layers=enc_layers)
        self.decoder = ManifoldDecoder(latent_dim, hidden_dim, input_ch, n_layers=dec_layers)
        self.head = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    @staticmethod
    def _masked_mean_time(z, mask=None, eps=1e-6):
        # z: (B,T,D)
        if mask is None:
            return z.mean(dim=1)
        m = mask.to(dtype=z.dtype, device=z.device).unsqueeze(-1)
        return (z * m).sum(dim=1) / (m.sum(dim=1) + eps)

    def forward(self, x, mask=None):
        z = self.encoder(x)              # (B,T,D)
        _ = self.decoder(z)              # (B,C,T) (kept for fairness; not used)
        z_pool = self._masked_mean_time(z, mask=mask)  # (B,D)
        y = self.head(z_pool).squeeze(-1)              # (B,)
        y = F.softplus(y)                               # >=0
        return y

    def forward_rate(self, x, mask=None, tau=1.0):
        # For prediction compatibility with windowing: return rate (count/duration)
        # But B0 is trained on window_count, so here we convert on the fly:
        # NOTE: duration should be provided externally; we will just return "count" here
        # and caller will divide by duration if needed. To keep predict_count_by_windowing
        # unchanged, we treat this as "rate" by dividing using mask length in this function.
        y_count = self.forward(x, mask=mask)  # (B,)
        # estimate duration from mask or x length
        if mask is None:
            T = x.shape[-1]
            dur = T / float(tau)  # tau unused, but keep signature; override below in predict wrapper
            # we cannot know fs here; caller uses predict_count_by_windowing which expects rate.
            # So we will not use this branch (we always pass mask=None but fixed win length).
            return y_count
        else:
            T = mask.sum(dim=1).clamp_min(1.0)
            return y_count / T  # in "count per sample" units (not used)


# ---------------------------------------------------------------------
# 8) Losses (OURS) + Train loops
# ---------------------------------------------------------------------
def masked_recon_mse(x_hat, x, mask, eps=1e-6):
    mask = mask.to(dtype=x.dtype, device=x.device)
    mask_bc = mask.unsqueeze(1)              # (B,1,T)
    se = (x_hat - x) ** 2                    # (B,C,T)
    se = se * mask_bc
    denom = (mask.sum() * x.shape[1]) + eps
    return se.sum() / denom


def temporal_smoothness(v, mask=None, eps=1e-6):
    dv = torch.abs(v[:, 1:] - v[:, :-1])
    if mask is None:
        return dv.mean()
    m = mask[:, 1:] * mask[:, :-1]
    m = m.to(dtype=dv.dtype, device=dv.device)
    return (dv * m).sum() / (m.sum() + eps)


def phase_entropy_loss(phase_p, mask=None, eps=1e-8):
    ent = -(phase_p * (phase_p + eps).log()).sum(dim=-1)  # (B,T)
    if mask is None:
        return ent.mean()
    ent = ent * mask
    return ent.sum() / (mask.sum() + eps)


def effK_usage_loss(phase_p, mask=None, eps=1e-6):
    if mask is None:
        p_bar = phase_p.mean(dim=1)
    else:
        m = mask.to(dtype=phase_p.dtype, device=phase_p.device).unsqueeze(-1)
        p_bar = (phase_p * m).sum(dim=1) / (m.sum(dim=1) + eps)
    effK = 1.0 / (p_bar.pow(2).sum(dim=1) + eps)
    return effK.mean()


def train_one_epoch_ours(model, loader, optimizer, config, device):
    model.train()
    fs = config["fs"]
    tau = config.get("tau", 1.0)

    lam_recon = config.get("lambda_recon", 1.0)
    lam_smooth = config.get("lambda_smooth", 0.05)
    lam_phase_ent = config.get("lambda_phase_ent", 0.01)
    lam_effk = config.get("lambda_effk", 0.0075)

    for batch in loader:
        x = batch["data"].to(device)
        mask = batch["mask"].to(device)
        y_count = batch["count"].to(device)
        length = batch["length"].to(device)

        duration = torch.clamp(length / fs, min=1e-6)
        y_rate = y_count / duration

        optimizer.zero_grad()
        rate_hat, aux = model(x, mask, tau=tau)

        loss_rate = F.mse_loss(rate_hat, y_rate)
        loss_recon = masked_recon_mse(aux["x_hat"], x, mask)
        loss_smooth = temporal_smoothness(aux["rep_rate_t"], mask)
        loss_phase_ent = phase_entropy_loss(aux["phase_p"], mask)
        loss_effk = effK_usage_loss(aux["phase_p"], mask)

        loss = (loss_rate
                + lam_recon * loss_recon
                + lam_smooth * loss_smooth
                + lam_phase_ent * loss_phase_ent
                + lam_effk * loss_effk)

        loss.backward()
        optimizer.step()


def train_one_epoch_b0(model, loader, optimizer, config, device):
    model.train()
    # B0 trains terminal window count regression: y_count
    for batch in loader:
        x = batch["data"].to(device)
        mask = batch["mask"].to(device)
        y_count = batch["count"].to(device)

        optimizer.zero_grad()
        y_hat = model(x, mask=mask)  # (B,) window_count_hat
        loss = F.mse_loss(y_hat, y_count)
        loss.backward()
        optimizer.step()


# ---------------------------------------------------------------------
# 9) Evaluation helpers
# ---------------------------------------------------------------------
def compute_errors(pred, gt):
    abs_err = abs(float(pred) - float(gt))
    mape = abs_err / (abs(float(gt)) + 1e-6) * 100.0
    bias = float(pred) - float(gt)
    return abs_err, mape, bias


def summarize_by_group(df, group_cols):
    g = (df.groupby(group_cols)
         .agg(
            n=("MAE", "count"),
            MAE=("MAE", "mean"),
            MAPE=("MAPE", "mean"),
            Bias=("Bias", "mean"),
         )
         .reset_index())
    return g


# ---------------------------------------------------------------------
# 10) Main Experiment: LOSO per activity, scenarios (orig/pause/drift)
# ---------------------------------------------------------------------
def run_pause_and_gradual_drift(CONFIG, full_data, device, ACTIVITY_SPECS):
    subjects = [f"subject{i}" for i in range(1, 11)]

    SCENARIOS = [
        ("orig", None),
        ("pause_insert", "pause"),
        ("gradual_tempo_drift", "drift"),
    ]

    all_rows = []

    print("\n" + "=" * 140)
    print(" >>> EXP-3) Constant-rate supervision risk defense")
    print(" >>> Compare: OURS vs B0 (loss-only change; B0 has bigger backbone)")
    print(" >>> TEST scenarios: orig | pause_insert | gradual_tempo_drift")
    print("=" * 140)

    for spec in ACTIVITY_SPECS:
        act_id = int(spec["act_id"])
        act_name = spec["act_name"]
        labels = spec["labels"]

        print("\n" + "-" * 140)
        print(f"[Activity] {act_name} (id={act_id}) | LOSO over 10 subjects")
        print("-" * 140)

        for fold_idx, test_subj in enumerate(subjects):
            set_strict_seed(CONFIG["seed"])

            train_labels = [t for t in labels if t[0] != test_subj]
            test_labels  = [t for t in labels if t[0] == test_subj]

            target_map_one = {act_id: act_name}
            feat_map_one   = {act_id: CONFIG["ACT_FEATURE_MAP"][act_id]}

            train_trials = prepare_trial_list(train_labels, full_data, target_map_one, feat_map_one)
            test_trials  = prepare_trial_list(test_labels,  full_data, target_map_one, feat_map_one)

            if len(train_trials) == 0 or len(test_trials) == 0:
                print(f"[Skip] Fold {fold_idx+1}: train_trials={len(train_trials)}, test_trials={len(test_trials)}")
                continue

            # Train windows (same for OURS / B0)
            train_windows = trial_list_to_windows(
                train_trials, fs=CONFIG["fs"],
                win_sec=CONFIG["win_sec"], stride_sec=CONFIG["stride_sec"],
                drop_last=CONFIG["drop_last"]
            )

            g = torch.Generator()
            g.manual_seed(CONFIG["seed"])

            train_loader = DataLoader(
                WindowDataset(train_windows),
                batch_size=CONFIG["batch_size"],
                shuffle=True,
                collate_fn=collate_variable_length,
                generator=g,
                num_workers=0
            )

            input_ch = train_windows[0]["data"].shape[1]

            # -----------------------------
            # Build models
            # -----------------------------
            ours = OursKAutoCount(
                input_ch=input_ch,
                hidden_dim=CONFIG["hidden_dim"],
                latent_dim=CONFIG["latent_dim"],
                K_max=CONFIG["K_max"],
                enc_layers=2,
                dec_layers=2
            ).to(device)

            b0 = B0TerminalRegressor(
                input_ch=input_ch,
                hidden_dim=CONFIG["hidden_dim"] * 2,
                latent_dim=CONFIG["latent_dim"],
                enc_layers=3,  # +1
                dec_layers=3   # +1
            ).to(device)

            opt_ours = torch.optim.Adam(ours.parameters(), lr=CONFIG["lr"])
            opt_b0   = torch.optim.Adam(b0.parameters(),   lr=CONFIG["lr"])

            sch_ours = torch.optim.lr_scheduler.StepLR(opt_ours, step_size=30, gamma=0.5)
            sch_b0   = torch.optim.lr_scheduler.StepLR(opt_b0,   step_size=30, gamma=0.5)

            # -----------------------------
            # Train both ONCE per fold
            # -----------------------------
            for epoch in range(CONFIG["epochs"]):
                train_one_epoch_ours(ours, train_loader, opt_ours, CONFIG, device)
                train_one_epoch_b0(b0,   train_loader, opt_b0,   CONFIG, device)
                sch_ours.step()
                sch_b0.step()

            ours.eval()
            b0.eval()

            # -----------------------------
            # Evaluate on test trial (single trial for that subject/activity)
            # -----------------------------
            item = test_trials[0]
            x0 = item["data"]
            gt = float(item["count"])

            base_seed = int(CONFIG["seed"] + (fold_idx + 1) * 100000)
            rng = np.random.RandomState(_stable_seed_from_text(item["meta"], base_seed))

            for scen_name, kind in SCENARIOS:
                if kind is None:
                    x = x0
                elif kind == "pause":
                    # fixed pause length; deterministic via seed for reproducibility if you want to randomize mid
                    x = perturb_pause_insert(
                        x0, fs=int(CONFIG["fs"]),
                        pause_sec=float(CONFIG["PAUSE_sec"]),
                        center_frac=float(CONFIG["PAUSE_center_frac"])
                    )
                elif kind == "drift":
                    # alternate slow/fast per fold for diversity (deterministic)
                    direction = "slow" if (fold_idx % 2 == 0) else "fast"
                    x = perturb_gradual_tempo_drift(
                        x0,
                        drift_strength=float(CONFIG["DRIFT_strength"]),
                        direction=direction
                    )
                else:
                    raise ValueError(kind)

                # OURS predicts count from rate
                pred_ours = predict_count_by_windowing(
                    model=ours,
                    x_np=x,
                    fs=CONFIG["fs"],
                    win_sec=CONFIG["win_sec"],
                    stride_sec=CONFIG["stride_sec"],
                    device=device,
                    tau=CONFIG.get("tau", 1.0),
                    batch_size=CONFIG.get("batch_size", 64)
                )

                # B0 predicts window_count; we need rate for windowing aggregator.
                # Easiest: implement a special windowing for B0 that averages count/window_dur -> rate.
                pred_b0 = predict_count_by_windowing_b0(
                    model=b0,
                    x_np=x,
                    fs=CONFIG["fs"],
                    win_sec=CONFIG["win_sec"],
                    stride_sec=CONFIG["stride_sec"],
                    device=device,
                    batch_size=CONFIG.get("batch_size", 64)
                )

                mae_o, mape_o, bias_o = compute_errors(pred_ours, gt)
                mae_b, mape_b, bias_b = compute_errors(pred_b0, gt)

                all_rows.append({
                    "act_id": act_id,
                    "act_name": act_name,
                    "fold": fold_idx + 1,
                    "test_subj": test_subj,
                    "scenario": scen_name,
                    "GT": gt,

                    "model": "OURS",
                    "Pred": float(pred_ours),
                    "MAE": float(mae_o),
                    "MAPE": float(mape_o),
                    "Bias": float(bias_o),
                })
                all_rows.append({
                    "act_id": act_id,
                    "act_name": act_name,
                    "fold": fold_idx + 1,
                    "test_subj": test_subj,
                    "scenario": scen_name,
                    "GT": gt,

                    "model": "B0",
                    "Pred": float(pred_b0),
                    "MAE": float(mae_b),
                    "MAPE": float(mape_b),
                    "Bias": float(bias_b),
                })

            # Fold quick print (MAE/MAPE averaged over 3 scenarios for sanity)
            df_fold = pd.DataFrame([r for r in all_rows if (r["act_id"] == act_id and r["fold"] == fold_idx + 1)])
            gfold = summarize_by_group(df_fold, ["model", "scenario"]).sort_values(["model", "scenario"])
            print(f"[Fold {fold_idx+1:2d}] Test={test_subj}")
            for _, rr in gfold.iterrows():
                print(f"  {rr['model']:4s} | {rr['scenario']:18s} | MAE={rr['MAE']:.3f} | MAPE={rr['MAPE']:.2f}% | n={int(rr['n'])}")

    return pd.DataFrame(all_rows)


# ---------------------------------------------------------------------
# 11) B0 Windowing prediction (count-head) -> mean rate -> total count
# ---------------------------------------------------------------------
def predict_count_by_windowing_b0(model, x_np, fs, win_sec, stride_sec, device, batch_size=64):
    """
    For B0:
      - model predicts window_count directly
      - convert to window_rate = window_count / window_duration
      - mean(window_rate) * total_duration -> total_count
    """
    win_len = int(round(win_sec * fs))
    stride = int(round(stride_sec * fs))
    T = x_np.shape[0]
    total_dur = T / float(fs)
    win_dur = win_len / float(fs)

    if T <= win_len:
        x_tensor = torch.tensor(x_np, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(device)  # (1,C,T)
        with torch.no_grad():
            y_count = model(x_tensor, mask=None)  # window_count
        # window_dur = total_dur here (since short trial)
        rate_hat = float(y_count.item()) / max(total_dur, 1e-6)
        return float(rate_hat * total_dur)

    starts = list(range(0, T - win_len + 1, stride))
    windows = np.stack([x_np[st:st + win_len] for st in starts], axis=0)  # (N,W,C)
    xw = torch.tensor(windows, dtype=torch.float32).permute(0, 2, 1).to(device)  # (N,C,W)

    cnts = []
    model.eval()
    with torch.no_grad():
        for i in range(0, xw.shape[0], batch_size):
            xb = xw[i:i + batch_size]
            y_cnt = model(xb, mask=None)  # (B,) window_count
            cnts.append(y_cnt.detach().cpu().numpy())

    cnts = np.concatenate(cnts, axis=0)  # (N,)
    rates = cnts / max(win_dur, 1e-6)
    rate_mean = float(rates.mean())
    return float(rate_mean * total_dur)


# ---------------------------------------------------------------------
# 12) Print summaries (only MAE/MAPE for OURS & B0)
# ---------------------------------------------------------------------
def print_final_summaries(df_all: pd.DataFrame):
    if df_all is None or len(df_all) == 0:
        print("[Warn] No rows to summarize.")
        return

    # Overall summary per scenario
    g_overall = (df_all.groupby(["model", "scenario"])
                .agg(
                    n=("MAE", "count"),
                    MAE_mean=("MAE", "mean"),
                    MAE_std=("MAE", "std"),
                    MAPE_mean=("MAPE", "mean"),
                    MAPE_std=("MAPE", "std"),
                )
                .reset_index()
                .sort_values(["scenario", "model"])
                .reset_index(drop=True))

    print("\n" + "=" * 140)
    print("[OVERALL] MAE / MAPE by scenario (across all activities & folds)")
    with pd.option_context('display.max_rows', 200, 'display.max_columns', 40, 'display.width', 200):
        print(g_overall.to_string(index=False))

    # Per-activity summary per scenario
    g_act = (df_all.groupby(["act_id", "act_name", "model", "scenario"])
            .agg(
                n=("MAE", "count"),
                MAE_mean=("MAE", "mean"),
                MAE_std=("MAE", "std"),
                MAPE_mean=("MAPE", "mean"),
                MAPE_std=("MAPE", "std"),
            )
            .reset_index()
            .sort_values(["act_id", "scenario", "model"])
            .reset_index(drop=True))

    print("\n" + "=" * 140)
    print("[PER-ACTIVITY] MAE / MAPE by scenario")
    with pd.option_context('display.max_rows', 999, 'display.max_columns', 50, 'display.width', 220):
        print(g_act.to_string(index=False))
    print("=" * 140)


# ---------------------------------------------------------------------
# 13) Main
# ---------------------------------------------------------------------
def main():
    BASE_CONFIG = {
        "seed": 42,
        "data_dir": "/content/drive/MyDrive/Colab Notebooks/HAR_data/MHEALTHDATASET",

        "COLUMN_NAMES": [
            'acc_chest_x', 'acc_chest_y', 'acc_chest_z',
            'ecg_1', 'ecg_2',
            'acc_ankle_x', 'acc_ankle_y', 'acc_ankle_z',
            'gyro_ankle_x', 'gyro_ankle_y', 'gyro_ankle_z',
            'mag_ankle_x', 'mag_ankle_y', 'mag_ankle_z',
            'acc_arm_x', 'acc_arm_y', 'acc_arm_z',
            'gyro_arm_x', 'gyro_arm_y', 'gyro_arm_z',
            'mag_arm_x', 'mag_arm_y', 'mag_arm_z',
            'activity_id'
        ],

        # training
        "epochs": 50,
        "lr": 5e-4,
        "batch_size": 64,
        "fs": 50,

        # windowing
        "win_sec": 8.0,
        "stride_sec": 4.0,
        "drop_last": True,

        # OURS model
        "hidden_dim": 128,
        "latent_dim": 16,
        "K_max": 6,

        # OURS loss weights
        "lambda_recon": 1.0,
        "lambda_smooth": 0.05,
        "lambda_phase_ent": 0.01,
        "lambda_effk": 0.0075,
        "tau": 1.0,

        # TEST perturb params (keep fixed, no tuning)
        "PAUSE_sec": 2.0,
        "PAUSE_center_frac": 0.5,
        "DRIFT_strength": 0.35,
    }

    DEFAULT_FEATS = [
        'acc_chest_x', 'acc_chest_y', 'acc_chest_z',
        'acc_ankle_x', 'acc_ankle_y', 'acc_ankle_z',
        'gyro_ankle_x', 'gyro_ankle_y', 'gyro_ankle_z',
        'acc_arm_x', 'acc_arm_y', 'acc_arm_z',
        'gyro_arm_x', 'gyro_arm_y', 'gyro_arm_z'
    ]

    ACTIVITY_SPECS = [
        {"act_id": 6,  "act_name": "Waist bends forward",       "labels": [
            ("subject1", 6, 21), ("subject2", 6, 19), ("subject3", 6, 21), ("subject4", 6, 20), ("subject5", 6, 20),
            ("subject6", 6, 20), ("subject7", 6, 20), ("subject8", 6, 21), ("subject9", 6, 21), ("subject10", 6, 20),
        ]},
        {"act_id": 7,  "act_name": "Frontal elevation of arms", "labels": [
            ("subject1", 7, 20), ("subject2", 7, 20), ("subject3", 7, 20), ("subject4", 7, 20), ("subject5", 7, 20),
            ("subject6", 7, 20), ("subject7", 7, 20), ("subject8", 7, 19), ("subject9", 7, 19), ("subject10", 7, 20),
        ]},
        {"act_id": 8,  "act_name": "Knees bending",             "labels": [
            ("subject1", 8, 20), ("subject2", 8, 21), ("subject3", 8, 21), ("subject4", 8, 19), ("subject5", 8, 20),
            ("subject6", 8, 20), ("subject7", 8, 21), ("subject8", 8, 21), ("subject9", 8, 21), ("subject10", 8, 21),
        ]},
        {"act_id": 12, "act_name": "Jump front & back",         "labels": [
            ("subject1", 12, 20), ("subject2", 12, 22), ("subject3", 12, 21), ("subject4", 12, 21), ("subject5", 12, 20),
            ("subject6", 12, 21), ("subject7", 12, 19), ("subject8", 12, 20), ("subject9", 12, 20), ("subject10", 12, 20),
        ]},
        {"act_id": 10, "act_name": "Jogging",                   "labels": [
            ("subject1", 10, 157), ("subject2", 10, 161), ("subject3", 10, 154), ("subject4", 10, 154), ("subject5", 10, 160),
            ("subject6", 10, 156), ("subject7", 10, 153), ("subject8", 10, 160), ("subject9", 10, 166), ("subject10", 10, 156),
        ]},
        {"act_id": 11, "act_name": "Running",                   "labels": [
            ("subject1", 11, 165), ("subject2", 11, 158), ("subject3", 11, 174), ("subject4", 11, 163), ("subject5", 11, 157),
            ("subject6", 11, 172), ("subject7", 11, 149), ("subject8", 11, 166), ("subject9", 11, 174), ("subject10", 11, 172),
        ]},
    ]

    target_map = {int(s["act_id"]): s["act_name"] for s in ACTIVITY_SPECS}
    feat_map = {int(s["act_id"]): DEFAULT_FEATS for s in ACTIVITY_SPECS}

    CONFIG = dict(BASE_CONFIG)
    CONFIG["TARGET_ACTIVITIES_MAP"] = target_map
    CONFIG["ACT_FEATURE_MAP"] = feat_map

    set_strict_seed(CONFIG["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    full_data = load_mhealth_dataset(CONFIG["data_dir"], CONFIG["TARGET_ACTIVITIES_MAP"], CONFIG["COLUMN_NAMES"])
    if not full_data:
        return

    df_all = run_pause_and_gradual_drift(CONFIG, full_data, device, ACTIVITY_SPECS)
    print_final_summaries(df_all)


if __name__ == "__main__":
    main()


Device: cuda

 >>> EXP-3) Constant-rate supervision risk defense
 >>> Compare: OURS vs B0 (loss-only change; B0 has bigger backbone)
 >>> TEST scenarios: orig | pause_insert | gradual_tempo_drift

--------------------------------------------------------------------------------------------------------------------------------------------
[Activity] Waist bends forward (id=6) | LOSO over 10 subjects
--------------------------------------------------------------------------------------------------------------------------------------------
[Fold  1] Test=subject1
  B0   | gradual_tempo_drift | MAE=21.000 | MAPE=100.00% | n=1
  B0   | orig               | MAE=21.000 | MAPE=100.00% | n=1
  B0   | pause_insert       | MAE=21.000 | MAPE=100.00% | n=1
  OURS | gradual_tempo_drift | MAE=3.589 | MAPE=17.09% | n=1
  OURS | orig               | MAE=2.126 | MAPE=10.13% | n=1
  OURS | pause_insert       | MAE=2.912 | MAPE=13.86% | n=1
[Fold  2] Test=subject2
  B0   | gradual_tempo_drift | MAE=19.000 |